<div style="display: flex; align-items: center; justify-content: space-between;">
    <img src="./venturepulse_logo.png" width="400" alt="VenturePulse Logo"/>
</div>

# VenturePulse — Intelligent Investor-Startup Matching Platform

### Big Data Analytics for Investment Optimization

---

## 📑 Table of Contents

### [Project Overview](#overview)
- Business Problem & Solution
- End-to-End Architecture  
- Technology Stack
- Datasets

### [Part 1: Structured Streaming](#part1)
Real-time startup registration ingestion
- [Step 1: Environment Setup](#step1)
- [Step 2: Load Investor Database](#step2)
- [Step 3: Kafka Source & Event Parsing](#step3)
- [Step 4: Data Storage (Parquet)](#step4)
- [Step 5: Real-Time Monitoring](#step5)
- [Step 6: Stop Streaming Queries](#step6)
- [Step 7: Verify Saved Data](#step7)

### [Part 2: Graph Processing](#part2)
Network analysis and investor influence
- [Step 8: GraphFrames Setup](#step8)
- [Step 9: Build Investor-Startup Network](#step9)
  - Create Vertices
  - Create Edges (Matching Criteria)
  - Construct Bipartite Graph
- [Step 10: Graph Analytics](#step10)
  - Degree Centrality
  - PageRank (Influential Investors)
  - Label Propagation (Communities)
- [Step 11: Network Insights](#step11)

### [Part 3: Machine Learning](#part3)
Predictive analytics for investment decisions
- [Step 12: ML Setup & Feature Engineering](#step12)
- [Step 13: Success Prediction Model](#step13)
  - Model Training
  - Model Evaluation
- [Step 14: Business Insights from ML](#step14)

### [Conclusion](#conclusion)

---

<a id='overview'></a>
## 📋 Project Overview

### Business Problem

The startup investment ecosystem faces critical inefficiencies:

**For Investors:**
- Difficulty discovering startups matching their sector and stage preferences
- Manual screening of hundreds of investment opportunities
- Limited visibility into startup network effects and influence

**For Startups:**
- Challenge identifying investors aligned with their funding needs
- Inefficient cold outreach to mismatched investors
- Lack of data-driven investor recommendations

### VenturePulse Solution

**VenturePulse** is an intelligent matching platform leveraging Big Data analytics to:

1. **Capture startup registrations in real-time** (Spark Structured Streaming)
2. **Build dynamic investor-startup networks** (Graph Processing)
3. **Generate data-driven recommendations** (Machine Learning)

### Architecture

```
┌─────────────────────┐
│   STARTUPS          │
│   Register on       │
│   Platform          │
└──────────┬──────────┘
           │
           ▼
┌─────────────────────┐       ┌─────────────────────┐
│   Apache Kafka      │──────▶│  Spark Streaming    │
│   (Event Broker)    │       │  (Real-time Ingest) │
└─────────────────────┘       └──────────┬──────────┘
                                         │
                                         ▼
                              ┌─────────────────────┐
                              │  Parquet Storage    │
                              │  (Data Lake)        │
                              └──────────┬──────────┘
                                         │
                   ┌─────────────────────┼─────────────────────┐
                   │                     │                     │
                   ▼                     ▼                     ▼
         ┌──────────────────┐  ┌──────────────────┐  ┌──────────────────┐
         │ GraphFrames      │  │ Spark ML         │  │ Batch Analytics  │
         │ (Network)        │  │ (Predictions)    │  │ (Reporting)      │
         └──────────────────┘  └──────────────────┘  └──────────────────┘
                   │                     │                     │
                   └─────────────────────┴─────────────────────┘
                                         │
                                         ▼
                              ┌─────────────────────┐
                              │  Business Insights  │
                              │  & Recommendations  │
                              └─────────────────────┘
```

### Technology Used

- **Streaming:** Apache Kafka + Spark Structured Streaming
- **Storage:** Parquet (columnar format)
- **Graph:** GraphFrames API
- **ML:** Spark MLlib 

### Key Datasets

1. **Investors** (500 records)
   - Investor profiles with sector preferences
   - Investment stage preferences (Seed, Series A/B/C)
   - Investment capacity (min/max check size)

2. **Startups** (streaming events)
   - Real-time registration events
   - Funding details (amount, stage, sector)
   - Company metrics (team size, growth rate)

---

<a id='part1'></a>
# Part 1: Structured Streaming

## Objective

Simulate real-time startup registrations on the VenturePulse platform and persist all events for subsequent batch processing.

**Why Streaming?**  
In a production environment with high platform engagement, startups register continuously. Streaming architecture enables:
- Real-time data ingestion at scale
- Immediate data availability for analytics
- Fault-tolerant event processing

---

<a id='step1'></a>
## Step 1: Environment Setup

Initialize Spark with required packages for Kafka integration.

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, DoubleType
import pandas as pd

# Configure ALL packages needed for the complete pipeline
os.environ['PYSPARK_SUBMIT_ARGS'] = '''
--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.2,graphframes:graphframes:0.8.3-spark3.5-s_2.12 pyspark-shell
'''.strip()

# Create Spark session (will load all packages)
spark = SparkSession.builder \
    .appName("VenturePulse - Complete Analytics Pipeline") \
    .getOrCreate()

print(f"This cluster relies on Spark '{spark.version}'")
print(f"   Application: {spark.sparkContext.appName}")
print(f"   Packages loaded:")
print(f"   - Kafka (Structured Streaming)")
print(f"   - GraphFrames (Network Analytics)")
print(f"   - Spark ML (Built-in, no package needed)")

26/03/07 18:49:21 WARN Utils: Your hostname, osbdet resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/07 18:49:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/osbdet/.jupyter_venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/osbdet/.ivy2/cache
The jars for the packages stored in: /home/osbdet/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-375221ef-8318-442d-965c-96f2b9108bb1;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.2 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
	found graphfram

This cluster relies on Spark '3.5.4'
   Application: VenturePulse - Complete Analytics Pipeline
   Packages loaded:
   - Kafka (Structured Streaming)
   - GraphFrames (Network Analytics)
   - Spark ML (Built-in, no package needed)


26/03/07 18:49:55 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


---

<a id='step2'></a>
## Step 2: Load Investor Database

Load the static investor database representing investors registered on VenturePulse.

**This dataset will be used for:**
- Building the investor-startup bipartite graph
- Defining matching criteria (sector, stage, investment capacity)
- Graph analytics (PageRank, centrality measures)

In [2]:
# Load investor database
investors_df = spark.read.csv(
    "investors.csv",
    header=True,
    inferSchema=True
)

# Cache for repeated access
investors_df.cache()

print(f"✅ Loaded {investors_df.count()} investors")
print("\nInvestor Profile Sample:")
investors_df.select(
    "investor_name",
    "investor_type",
    "sectors_preference",
    "investment_stage_preference"
).show(5, truncate=False)

DataFrame[investor_id: string, investor_name: string, investor_type: string, location: string, sectors_preference: string, min_check_size_usd: int, max_check_size_usd: int, portfolio_companies: int, founded_year: int, investment_stage_preference: string]

✅ Loaded 500 investors

Investor Profile Sample:
+----------------+-------------+----------------------------+---------------------------+
|investor_name   |investor_type|sectors_preference          |investment_stage_preference|
+----------------+-------------+----------------------------+---------------------------+
|Stellar Capital |Accelerator  |Web3;5G                     |Series D;Series A;Series C |
|Alpha Collective|Angel        |E-commerce;SaaS             |Series C;Series A;Seed     |
|Prime Labs      |Angel Network|SaaS;Web3;HealthTech;5G     |Series C;Series D;Seed     |
|Next Equity     |Angel Network|5G;EdTech;Fintech;E-commerce|Series B;Series C          |
|Apex Capital    |Angel Network|SaaS;EdTech;FoodTech        |Series A                   |
+----------------+-------------+----------------------------+---------------------------+
only showing top 5 rows



---

## ⚠️ IMPORTANT: Start Event Generator

**Before proceeding, open a terminal and run:**

```bash
python3 startup_event_generator.py
```

This script simulates startups registering on VenturePulse by producing events to Kafka topic `funding_event`.

**Wait for confirmation:** `[5s] Sent 10 events | Partition: 0 | Offset: 10`

Then continue to the next cell.

---

<a id='step3'></a>
## Step 3: Kafka Source & Event Parsing

Connect to Kafka and parse incoming startup registration events.

**Event Format** (pipe-delimited):
```
timestamp|category|event_id|startup_id|startup_name|sector|stage|amount|location|team|...
```

In [3]:
# Read from Kafka topic
raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "funding_event") \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

# Parse pipe-delimited events
startup_events_df = raw_stream \
    .select(F.split(F.col("value").cast("string"), "\\|").alias("fields")) \
    .select(
        # Timestamp for event-time processing
        F.to_timestamp(F.col("fields")[0], "yyyy-MM-dd HH:mm:ss.SSS").alias("eventTime"),
        
        # Event metadata
        F.col("fields")[1].alias("event_category"),
        F.col("fields")[2].cast(IntegerType()).alias("event_id"),
        
        # Startup identification (critical for graph construction)
        F.col("fields")[3].alias("startup_id"),
        F.col("fields")[4].alias("startup_name"),
        
        # Matching criteria
        F.col("fields")[5].alias("sector"),
        F.col("fields")[6].alias("funding_stage"),
        F.col("fields")[7].cast(LongType()).alias("funding_amount_usd"),
        F.col("fields")[8].alias("location"),
        
        # Company metrics
        F.col("fields")[9].cast(IntegerType()).alias("team_size"),
        F.col("fields")[10].cast(IntegerType()).alias("months_since_founded"),
        F.col("fields")[11].cast(LongType()).alias("revenue_usd"),
        F.col("fields")[12].cast(LongType()).alias("monthly_recurring_revenue"),
        F.col("fields")[13].cast(DoubleType()).alias("team_growth_rate_6mo"),
        F.col("fields")[14].alias("event_type")
    )

print("   Kafka source configured")
print("   Topic: funding_event")
print("   Events parsed with 15 fields")

   Kafka source configured
   Topic: funding_event
   Events parsed with 15 fields


---

<a id='step4'></a>
## Step 4: Data Storage (Parquet)

Persist all startup events to Parquet for batch processing.

**Why Parquet?**
- Columnar storage format optimized for analytics
- Efficient compression and encoding


In [4]:
# Write all events to Parquet
storage_path = "/tmp/venturepulse/all_startup_events"
checkpoint_path = "/tmp/venturepulse/checkpoints/storage"

storage_query = startup_events_df \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", storage_path) \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(processingTime="30 seconds") \
    .start()

print("✅ Storage query started")
print(f"   Destination: {storage_path}")


26/03/07 18:51:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


✅ Storage query started
   Destination: /tmp/venturepulse/all_startup_events


26/03/07 18:51:13 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
[Stage 7:>                                                          (0 + 1) / 1]

---

<a id='step5'></a>
## Step 5: Real-Time Monitoring

Display individual startup registrations as they occur.

**Purpose:** Verify data quality and observe platform activity in real-time.

In [5]:
# Console output for monitoring
monitor_query = startup_events_df \
    .select(
        "eventTime",
        "startup_id",
        "startup_name",
        "sector",
        "funding_stage",
        F.format_number("funding_amount_usd", 0).alias("funding_usd"),
        "location"
    ) \
    .writeStream \
    .format("console") \
    .outputMode("append") \
    .option("truncate", "false") \
    .option("numRows", 10) \
    .trigger(processingTime="5 seconds") \
    .start()

print("✅ Real-time monitoring started")
print("   Updates every 5 seconds")
print("\n⏱️  Let queries run for 1-2 minutes to collect data...")

26/03/07 18:51:19 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8df21472-f3fd-4c34-b5e2-faab63c6479e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/03/07 18:51:19 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


✅ Real-time monitoring started
   Updates every 5 seconds

⏱️  Let queries run for 1-2 minutes to collect data...


26/03/07 18:51:19 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------------------+----------+-------------+----------+-------------+-----------+---------+
|eventTime              |startup_id|startup_name |sector    |funding_stage|funding_usd|location |
+-----------------------+----------+-------------+----------+-------------+-----------+---------+
|2026-03-07 18:51:20.412|STR_00047 |BetaSolutions|Gaming    |Seed         |1,008,647  |Amsterdam|
|2026-03-07 18:51:20.934|STR_00048 |MetaStream   |E-commerce|Series A     |14,709,265 |Barcelona|
+-----------------------+----------+-------------+----------+-------------+-----------+---------+



26/03/07 18:51:24 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 5449 milliseconds
                                                                                

...

-------------------------------------------
Batch: 33
-------------------------------------------
+-----------------------+----------+----------------+----------+-------------+-----------+---------+
|eventTime              |startup_id|startup_name    |sector    |funding_stage|funding_usd|location |
+-----------------------+----------+----------------+----------+-------------+-----------+---------+
|2026-03-07 18:53:55.47 |STR_00349 |CyberInnovations|HealthTech|Series B     |48,112,539 |Paris    |
|2026-03-07 18:53:55.977|STR_00350 |DataFlow        |AI        |Series C     |36,456,381 |Boston   |
|2026-03-07 18:53:56.488|STR_00351 |DigiWorks       |HealthTech|Seed         |679,927    |London   |
|2026-03-07 18:53:56.995|STR_00352 |NanoAI          |E-commerce|Seed         |807,805    |Berlin   |
|2026-03-07 18:53:57.501|STR_00353 |EcoVertex       |IoT       |Series B     |47,538,428 |Austin   |
|2026-03-07 18:53:58.005|STR_00354 |BioTech         |Mobility  |Series C     |86,396,039 |Berl

-------------------------------------------
Batch: 38
-------------------------------------------
+-----------------------+----------+--------------+----------+-------------+-----------+---------+
|eventTime              |startup_id|startup_name  |sector    |funding_stage|funding_usd|location |
+-----------------------+----------+--------------+----------+-------------+-----------+---------+
|2026-03-07 18:54:20.351|STR_00398 |EcoHub        |Web3      |Series A     |14,368,998 |Barcelona|
|2026-03-07 18:54:20.863|STR_00399 |RapidTech     |Web3      |Series A     |3,701,519  |Boston   |
|2026-03-07 18:54:21.368|STR_00400 |PrimeSystems  |Fintech   |Series C     |30,615,179 |Paris    |
|2026-03-07 18:54:21.874|STR_00401 |NeuroNexus    |CleanTech |Seed         |1,002,812  |Berlin   |
|2026-03-07 18:54:22.381|STR_00402 |BioVertex     |IoT       |Series B     |38,648,375 |Stockholm|
|2026-03-07 18:54:22.885|STR_00403 |BioAI         |SaaS      |Seed         |463,261    |Chicago  |
|2026-03-07

-------------------------------------------
Batch: 49
-------------------------------------------
+-----------------------+----------+-------------+----------+-------------+-----------+-------------+
|eventTime              |startup_id|startup_name |sector    |funding_stage|funding_usd|location     |
+-----------------------+----------+-------------+----------+-------------+-----------+-------------+
|2026-03-07 18:55:15.26 |STR_00506 |EcoPlatform  |Web3      |Series C     |69,110,409 |San Francisco|
|2026-03-07 18:55:15.773|STR_00507 |CyberVertex  |IoT       |Seed         |426,193    |Stockholm    |
|2026-03-07 18:55:16.28 |STR_00508 |RapidNetwork |E-commerce|Series B     |13,005,468 |Stockholm    |
|2026-03-07 18:55:16.786|STR_00509 |CoreSolutions|HealthTech|Series B     |43,011,316 |Dublin       |
|2026-03-07 18:55:17.294|STR_00510 |RoboPulse    |Mobility  |Seed         |647,611    |Stockholm    |
|2026-03-07 18:55:17.802|STR_00511 |NeuroSync    |Mobility  |Series A     |8,977,842  

-------------------------------------------
Batch: 52
-------------------------------------------
+-----------------------+----------+---------------+----------+-------------+-----------+---------+
|eventTime              |startup_id|startup_name   |sector    |funding_stage|funding_usd|location |
+-----------------------+----------+---------------+----------+-------------+-----------+---------+
|2026-03-07 18:55:30.554|STR_00536 |HyperAnalytics |IoT       |Series A     |13,263,749 |New York |
|2026-03-07 18:55:31.082|STR_00537 |TechSolutions  |Fintech   |Series B     |20,207,615 |Berlin   |
|2026-03-07 18:55:31.599|STR_00538 |NeuroNexus     |SaaS      |Series A     |6,192,917  |Miami    |
|2026-03-07 18:55:32.105|STR_00539 |FutureAnalytics|AI        |Series C     |148,425,294|Stockholm|
|2026-03-07 18:55:32.61 |STR_00540 |HyperHub       |IoT       |Series A     |11,854,915 |Stockholm|
|2026-03-07 18:55:33.116|STR_00541 |AutoTech       |CleanTech |Series C     |116,624,915|London   |
|2

-------------------------------------------
Batch: 57
-------------------------------------------
+-----------------------+----------+---------------+----------+-------------+-----------+-----------+
|eventTime              |startup_id|startup_name   |sector    |funding_stage|funding_usd|location   |
+-----------------------+----------+---------------+----------+-------------+-----------+-----------+
|2026-03-07 18:55:55.558|STR_00585 |RoboNexus      |HealthTech|Series A     |14,322,272 |New York   |
|2026-03-07 18:55:56.075|STR_00586 |AlphaLabs      |Fintech   |Seed         |834,437    |Toronto    |
|2026-03-07 18:55:56.581|STR_00587 |FutureSolutions|IoT       |Series A     |13,798,035 |Los Angeles|
+-----------------------+----------+---------------+----------+-------------+-----------+-----------+



---

<a id='step6'></a>
## Step 6: Stop Streaming Queries

After collecting sufficient data, stop all active streaming queries.

In [6]:
# Stop all queries
print("Stopping streaming queries...")
print("-" * 60)

for query in [storage_query, monitor_query]:
    if query.isActive:
        query.stop()
        print(f"✅ Query stopped")

print("-" * 60)
print("✅ All streaming queries stopped")

Stopping streaming queries...
------------------------------------------------------------
✅ Query stopped
✅ Query stopped
------------------------------------------------------------
✅ All streaming queries stopped


---

<a id='step7'></a>
## Step 7: Verify Saved Data

Confirm data persistence and preview the dataset for graph processing.

In [8]:
# Read saved Parquet files
startups_df = spark.read.parquet("/tmp/venturepulse/all_startup_events")

total_events = startups_df.count()
print(f"✅ Successfully saved {total_events} startup events")
print("\nData Summary:")
startups_df.groupBy("sector").count().orderBy(F.desc("count")).show(10)

print("\nSample Records:")
startups_df.select(
    "startup_id",
    "startup_name",
    "sector",
    "funding_stage",
    "funding_amount_usd"
).show(10, truncate=False)

print("\n" + "="*60)
print("✅ Part 1: Streaming Complete")
print("="*60)
print(f"\nDataset Ready: {total_events} startups")
print("="*60)

✅ Successfully saved 1634 startup events

Data Summary:
+----------+-----+
|    sector|count|
+----------+-----+
|       IoT|  179|
|      SaaS|  174|
|    Gaming|  169|
|   Fintech|  169|
| CleanTech|  165|
|      Web3|  163|
|HealthTech|  160|
|        AI|  160|
|  Mobility|  152|
|E-commerce|  143|
+----------+-----+


Sample Records:


[Stage 91:>                                                         (0 + 1) / 1]

+----------+---------------+----------+-------------+------------------+
|startup_id|startup_name   |sector    |funding_stage|funding_amount_usd|
+----------+---------------+----------+-------------+------------------+
|STR_00476 |NanoDynamics   |E-commerce|Series C     |96546386          |
|STR_00477 |MetaAnalytics  |IoT       |Series C     |62694625          |
|STR_00478 |NeuroLabs      |Web3      |Series C     |128252230         |
|STR_00479 |CyberSystems   |E-commerce|Seed         |1642756           |
|STR_00480 |AlphaStream    |Mobility  |Series A     |12581337          |
|STR_00481 |FutureWorks    |HealthTech|Series C     |133691817         |
|STR_00482 |AlphaWorks     |CleanTech |Seed         |1113149           |
|STR_00483 |NanoSystems    |IoT       |Series C     |44780519          |
|STR_00484 |EcoDynamics    |Mobility  |Series A     |9369412           |
|STR_00485 |DigiInnovations|SaaS      |Seed         |732015            |
+----------+---------------+----------+------------

## 📊 STREAMING INSIGHT 1: SECTOR DISTRIBUTION

Understanding which industries dominate the startup ecosystem helps investors identify market trends and concentrate resources in high-activity sectors. This analysis reveals the sector landscape and highlights where the majority of startup innovation is occurring.

In [9]:
print("=" * 70)
print("INSIGHT 1: SECTOR DISTRIBUTION")
print("=" * 70)

sector_dist = startups_df.groupBy("sector") \
    .count() \
    .orderBy(F.desc("count"))

print("\nStartups by Sector:")
sector_dist.show(truncate=False)

# Top 3
top_3_sectors = sector_dist.limit(3).collect()
print(f"\n💡 Key Insight: Top 3 sectors represent hot investment areas:")
for i, row in enumerate(top_3_sectors, 1):
    pct = (row['count'] / startups_df.count()) * 100
    print(f"   {i}. {row['sector']}: {row['count']} startups ({pct:.1f}%)")

INSIGHT 1: SECTOR DISTRIBUTION

Startups by Sector:
+----------+-----+
|sector    |count|
+----------+-----+
|IoT       |179  |
|SaaS      |174  |
|Gaming    |169  |
|Fintech   |169  |
|CleanTech |165  |
|Web3      |163  |
|HealthTech|160  |
|AI        |160  |
|Mobility  |152  |
|E-commerce|143  |
+----------+-----+


💡 Key Insight: Top 3 sectors represent hot investment areas:
   1. IoT: 179 startups (11.0%)
   2. SaaS: 174 startups (10.6%)
   3. Fintech: 169 startups (10.3%)


## 📈 STREAMING INSIGHT 2: FUNDING STAGE MATURITY

Analyzing the distribution of startups across funding stages reveals the ecosystem's maturity and growth trajectory. This insight distinguishes between early-stage ventures (Seed) seeking initial validation and growth-stage companies (Series B/C) scaling their operations, helping investors align their risk appetite with appropriate investment stages.

In [10]:
print("\n" + "=" * 70)
print("INSIGHT 2: FUNDING STAGE MATURITY")
print("=" * 70)

stage_dist = startups_df.groupBy("funding_stage") \
    .count() \
    .orderBy(F.desc("count"))

print("\nStartups by Funding Stage:")
stage_dist.show(truncate=False)

total = startups_df.count()
early_stage = startups_df.filter(F.col("funding_stage") == "Seed").count()
growth_stage = startups_df.filter(F.col("funding_stage").isin(["Series B", "Series C"])).count()

print(f"\n💡 Key Insight: Stage distribution reflects ecosystem maturity:")
print(f"   Early-stage (Seed): {early_stage} startups ({early_stage/total*100:.1f}%)")
print(f"   Growth-stage (B/C): {growth_stage} startups ({growth_stage/total*100:.1f}%)")


INSIGHT 2: FUNDING STAGE MATURITY

Startups by Funding Stage:


+-------------+-----+
|funding_stage|count|
+-------------+-----+
|Series A     |418  |
|Seed         |414  |
|Series B     |412  |
|Series C     |390  |
+-------------+-----+




💡 Key Insight: Stage distribution reflects ecosystem maturity:
   Early-stage (Seed): 414 startups (25.3%)
   Growth-stage (B/C): 802 startups (49.1%)


## 💰 STREAMING INSIGHT 3: CAPITAL REQUIREMENTS

Understanding funding patterns and capital requirements is critical for matching investor capacity with startup needs. This analysis examines the range, average, and total capital being sought, providing insights into the financial scale of investment opportunities and helping investors plan portfolio allocation strategies.


In [11]:
print("\n" + "=" * 70)
print("INSIGHT 3: CAPITAL REQUIREMENTS")
print("=" * 70)

funding_stats = startups_df.select(
    F.min("funding_amount_usd").alias("min_funding"),
    F.max("funding_amount_usd").alias("max_funding"),
    F.avg("funding_amount_usd").alias("avg_funding"),
    F.sum("funding_amount_usd").alias("total_capital_seeking")
).first()

print(f"\nFunding Amount Statistics:")
print(f"   Range: ${funding_stats['min_funding']:,.0f} - ${funding_stats['max_funding']:,.0f}")
print(f"   Average: ${funding_stats['avg_funding']:,.0f}")
print(f"   Total Capital Seeking: ${funding_stats['total_capital_seeking']:,.0f}")

# By stage
avg_by_stage = startups_df.groupBy("funding_stage") \
    .agg(F.avg("funding_amount_usd").alias("avg_amount")) \
    .orderBy(F.desc("avg_amount"))

print("\nAverage Funding by Stage:")
avg_by_stage.show(truncate=False)

print(f"\n💡 Key Insight: Total ${funding_stats['total_capital_seeking']/1000000:.0f}M capital seeking investment")


INSIGHT 3: CAPITAL REQUIREMENTS

Funding Amount Statistics:
   Range: $307,001 - $149,467,845
   Average: $30,799,444
   Total Capital Seeking: $50,326,291,116

Average Funding by Stage:


+-------------+--------------------+
|funding_stage|avg_amount          |
+-------------+--------------------+
|Series C     |8.694111703589743E7 |
|Series B     |2.9814375475728154E7|
|Series A     |8747041.437799044   |
|Seed         |1158138.7801932367  |
+-------------+--------------------+


💡 Key Insight: Total $50326M capital seeking investment


## 👥 STREAMING INSIGHT 4: COMPANY TRACTION METRICS

Team dynamics and growth indicators serve as powerful predictors of startup success. This analysis examines team size, growth rates, and company age to identify startups with strong operational momentum and the human capital necessary for scaling, revealing which ventures show signs of sustainable expansion.

In [12]:
print("\n" + "=" * 70)
print("INSIGHT 4: COMPANY TRACTION METRICS")
print("=" * 70)

team_stats = startups_df.select(
    F.avg("team_size").alias("avg_team"),
    F.avg("team_growth_rate_6mo").alias("avg_growth"),
    F.avg("months_since_founded").alias("avg_age")
).first()

print(f"\nTraction Indicators:")
print(f"   Average Team Size: {team_stats['avg_team']:.1f} people")
print(f"   Average Growth Rate: {team_stats['avg_growth']:.1%}")
print(f"   Average Company Age: {team_stats['avg_age']:.0f} months")

# High-growth startups
high_growth = startups_df.filter(F.col("team_growth_rate_6mo") > 0.3).count()
pct_high_growth = (high_growth / startups_df.count()) * 100

print(f"\n💡 Key Insight: {high_growth} startups ({pct_high_growth:.1f}%) show high growth (>30% team expansion)")


INSIGHT 4: COMPANY TRACTION METRICS



Traction Indicators:
   Average Team Size: 78.7 people
   Average Growth Rate: 22.5%
   Average Company Age: 33 months

💡 Key Insight: 451 startups (27.6%) show high growth (>30% team expansion)


## 🌍 STREAMING INSIGHT 5: GEOGRAPHIC DISTRIBUTION

Geographic concentration patterns reveal startup ecosystem hotspots and enable location-based investment strategies. Understanding where startups cluster helps investors leverage local networks, access regional expertise, and match opportunities with geographically-focused investment mandates or local market knowledge.

In [13]:
print("\n" + "=" * 70)
print("INSIGHT 5: GEOGRAPHIC DISTRIBUTION")
print("=" * 70)

location_dist = startups_df.groupBy("location") \
    .count() \
    .orderBy(F.desc("count"))

print("\nTop 10 Startup Hubs:")
location_dist.show(10, truncate=False)

total_locations = location_dist.count()
top_3_locations = location_dist.limit(3).collect()
top_3_count = sum([row['count'] for row in top_3_locations])
concentration = (top_3_count / startups_df.count()) * 100

print(f"\n💡 Key Insight:")
print(f"   Total locations: {total_locations} cities")
print(f"   Top 3 hubs concentration: {concentration:.1f}% of startups")
print(f"   Geographic diversity enables local investor matching")


INSIGHT 5: GEOGRAPHIC DISTRIBUTION

Top 10 Startup Hubs:
+-------------+-----+
|location     |count|
+-------------+-----+
|Austin       |102  |
|Singapore    |91   |
|Palo Alto    |90   |
|Paris        |89   |
|Los Angeles  |87   |
|Dublin       |86   |
|Chicago      |86   |
|New York     |86   |
|Amsterdam    |85   |
|San Francisco|84   |
+-------------+-----+
only showing top 10 rows




💡 Key Insight:
   Total locations: 20 cities
   Top 3 hubs concentration: 17.3% of startups
   Geographic diversity enables local investor matching


---

<a id='part2'></a>
# Part 2: Graph Processing

## Objective

Build an investor-startup bipartite graph to identify optimal matches and discover network insights.

**Key Questions:**
1. Which investors have the most investment opportunities?
2. Which startups have the most potential investor matches?
3. Who are the most influential investors in the network?
4. Are there investment communities or clusters?
5. Which startups/investors are isolated (no matches)?

**Graph Structure:**
```
Investor A ─────▶ Startup X
     │
     ├──────────▶ Startup Y
     │
     └──────────▶ Startup Z

Investor B ─────▶ Startup Y
     │
     └──────────▶ Startup W
```

**Edge Creation Criteria:**
- Sector match (startup.sector IN investor.sectors_preference)
- Stage match (startup.funding_stage IN investor.investment_stage_preference)
- Amount fit (startup.funding_amount BETWEEN investor.min_check AND max_check)

---

<a id='step8'></a>
## Step 8: GraphFrames Setup

Configure Spark to load the GraphFrames package.

---

<a id='step9'></a>
## Step 9: Build Investor-Startup Network

### 9.1 Reload Datasets

Load both investors and startups for graph construction.

In [14]:
# Reload investors
investors_df = spark.read.csv("investors.csv", header=True, inferSchema=True)

# Reload startups from Parquet
startups_df = spark.read.parquet("/tmp/venturepulse/all_startup_events")

print(f"✅ Loaded {investors_df.count()} investors")
print(f"✅ Loaded {startups_df.count()} startups")

✅ Loaded 500 investors
✅ Loaded 1634 startups


### 9.2 Create Vertices DataFrame

Combine investors and startups into a single vertices DataFrame.

**Vertex Schema:**
- `id` (string): Unique identifier
- `name` (string): Display name
- `type` (string): "investor" or "startup"
- Additional attributes for analysis

In [15]:
# Investor vertices
investor_vertices = investors_df.select(
    F.col("investor_id").alias("id"),
    F.col("investor_name").alias("name"),
    F.lit("investor").alias("type"),
    "investor_type",
    "sectors_preference",
    "investment_stage_preference",
    "min_check_size_usd",
    "max_check_size_usd",
    "portfolio_companies"
)

# Startup vertices
startup_vertices = startups_df.select(
    F.col("startup_id").alias("id"),
    F.col("startup_name").alias("name"),
    F.lit("startup").alias("type"),
    "sector",
    "funding_stage",
    "funding_amount_usd",
    "location",
    "team_size"
).dropDuplicates(["id"])  # Remove duplicate startups

print(f"✅ Created {investor_vertices.count()} investor vertices")
print(f"✅ Created {startup_vertices.count()} startup vertices")

print("\nSample Investor Vertex:")
investor_vertices.limit(1).show(truncate=False)

print("Sample Startup Vertex:")
startup_vertices.limit(1).show(truncate=False)

✅ Created 500 investor vertices
✅ Created 587 startup vertices

Sample Investor Vertex:
+-------+---------------+--------+-------------+------------------+---------------------------+------------------+------------------+-------------------+
|id     |name           |type    |investor_type|sectors_preference|investment_stage_preference|min_check_size_usd|max_check_size_usd|portfolio_companies|
+-------+---------------+--------+-------------+------------------+---------------------------+------------------+------------------+-------------------+
|INV_001|Stellar Capital|investor|Accelerator  |Web3;5G           |Series D;Series A;Series C |50303             |798050            |134                |
+-------+---------------+--------+-------------+------------------+---------------------------+------------------+------------------+-------------------+

Sample Startup Vertex:


+---------+---------+-------+-------+-------------+------------------+--------+---------+
|id       |name     |type   |sector |funding_stage|funding_amount_usd|location|team_size|
+---------+---------+-------+-------+-------------+------------------+--------+---------+
|STR_00001|PrimeSync|startup|Fintech|Series B     |45113114          |Tel Aviv|85       |
+---------+---------+-------+-------+-------------+------------------+--------+---------+



### 9.3 Create Edges DataFrame

Generate edges based on matching criteria.

**Edge represents:** An investor CAN invest in a startup (match exists)

**Matching Logic:**
1. Sector alignment
2. Stage alignment  
3. Funding amount within investor's capacity

In [16]:
# Cross join investors with startups
potential_matches = investors_df.crossJoin(startups_df)

# Apply matching criteria
matches = potential_matches.filter(
    (F.array_contains(F.split(F.col("sectors_preference"), ";"), F.col("sector"))) &
    (F.array_contains(F.split(F.col("investment_stage_preference"), ";"), F.col("funding_stage"))) &
    (F.col("funding_amount_usd") >= F.col("min_check_size_usd")) &
    (F.col("funding_amount_usd") <= F.col("max_check_size_usd"))
)

# Create edges in BOTH directions
investor_to_startup = matches.select(
    F.col("investor_id").alias("src"),
    F.col("startup_id").alias("dst"),
    F.lit("can_invest").alias("relationship")
)

startup_to_investor = matches.select(
    F.col("startup_id").alias("src"),
    F.col("investor_id").alias("dst"),
    F.lit("seeks_investment").alias("relationship")
)

# Union both directions
edges_df = investor_to_startup.union(startup_to_investor)

print(f"✅ Created {edges_df.count()} edges (bidirectional)")
print(f"   - {investor_to_startup.count()} investor → startup")
print(f"   - {startup_to_investor.count()} startup → investor")
    


✅ Created 55646 edges (bidirectional)


   - 27823 investor → startup


[Stage 181:============================>                            (1 + 1) / 2]

   - 27823 startup → investor


### 9.4 Union Vertices and Construct GraphFrame

Combine investor and startup vertices, then create the GraphFrame.

In [17]:
# Union vertices (align schemas first)
from graphframes import GraphFrame
from pyspark.sql.functions import lit

# Add missing columns to align schemas
investor_v = investor_vertices.select(
    "id", "name", "type",
    lit(None).cast("string").alias("sector"),
    lit(None).cast("string").alias("funding_stage"),
    lit(None).cast("long").alias("funding_amount_usd")
)

startup_v = startup_vertices.select(
    "id", "name", "type",
    "sector",
    "funding_stage",
    "funding_amount_usd"
)

# Union all vertices
vertices_df = investor_v.union(startup_v)

# Create GraphFrame
venturepulse_graph = GraphFrame(vertices_df, edges_df)
venturepulse_graph.cache()

print("✅ GraphFrame constructed")
print(f"   Total vertices: {venturepulse_graph.vertices.count()}")
print(f"   Total edges: {venturepulse_graph.edges.count()}")
print(f"\n📊 Network Density: {edges_df.count() / (investors_df.count() * startup_vertices.count()) * 100:.2f}%")

GraphFrame(v:[id: string, name: string ... 4 more fields], e:[src: string, dst: string ... 1 more field])

✅ GraphFrame constructed


   Total vertices: 1087


   Total edges: 55646

📊 Network Density: 18.96%


---

<a id='step10'></a>
## Step 10: Graph Analytics

Apply graph algorithms to extract business insights.

### 10.1 Degree Centrality Analysis

**Question:** Which investors have the most opportunities? Which startups have the most options?

**Metric:** Out-degree (investors) and In-degree (startups)

In [18]:
# Calculate degrees
degrees = venturepulse_graph.degrees
in_degrees = venturepulse_graph.inDegrees
out_degrees = venturepulse_graph.outDegrees

# Top investors by opportunities (out-degree)
print("=" * 70)
print("TOP 10 INVESTORS BY INVESTMENT OPPORTUNITIES")
print("=" * 70)
top_investors = out_degrees \
    .join(venturepulse_graph.vertices, "id") \
    .filter(F.col("type") == "investor") \
    .select("name", F.col("outDegree").alias("potential_investments")) \
    .orderBy(F.desc("potential_investments"))

top_investors.limit(10).toPandas()

print("\n" + "=" * 70)
print("TOP 10 STARTUPS BY INVESTOR MATCHES")
print("=" * 70)
top_startups = in_degrees \
    .join(venturepulse_graph.vertices, "id") \
    .filter(F.col("type") == "startup") \
    .select("name", "sector", "funding_stage", F.col("inDegree").alias("investor_matches")) \
    .orderBy(F.desc("investor_matches"))

top_startups.limit(10).show(truncate=False)

top_sector = top_startups.groupBy("sector") \
    .agg(F.avg("investor_matches").alias("avg_matches")) \
    .orderBy(F.desc("avg_matches")) \
    .first()

print(f"\n💡 Insights:")
print(f"   - Sector with highest avg matches: {top_sector['sector']} ({top_sector['avg_matches']:.1f} investors/startup)")
print(f"   - Degree centrality reveals startups with most investment options")
print(f"   - High-degree startups may face competitive bidding from investors")

TOP 10 INVESTORS BY INVESTMENT OPPORTUNITIES


,name,potential_investments
0,BetaGrowth,356
1,NextLabs,319
2,StellarInnovation,273
3,HorizonFund,253
4,PeakHoldings,246
5,GammaCapital,239
6,BetaGrowth,237
7,BetaLabs,237
8,ZenithInnovation,234
9,Future Investment,230



TOP 10 STARTUPS BY INVESTOR MATCHES


+--------------+----------+-------------+----------------+
|name          |sector    |funding_stage|investor_matches|
+--------------+----------+-------------+----------------+
|DigiFlow      |SaaS      |Series B     |91              |
|AutoAnalytics |Mobility  |Series A     |89              |
|RoboTech      |E-commerce|Series A     |86              |
|BetaLabs      |IoT       |Series B     |85              |
|NeuroSolutions|Fintech   |Series A     |84              |
|TechVertex    |SaaS      |Series C     |84              |
|BetaStream    |E-commerce|Series B     |83              |
|NeuroSync     |Mobility  |Series A     |83              |
|DigiWorks     |HealthTech|Series B     |81              |
|TechSolutions |E-commerce|Series A     |80              |
+--------------+----------+-------------+----------------+




💡 Insights:
   - Sector with highest avg matches: E-commerce (50.2 investors/startup)
   - Degree centrality reveals startups with most investment options
   - High-degree startups may face competitive bidding from investors


### 10.2 PageRank: Identify Influential Investors

**Question:** Who are the most influential investors in the network?

**Algorithm:** PageRank (originally developed for web page ranking)

**Interpretation:** High PageRank = High influence (connected to well-connected startups)

In [19]:
# Run PageRank
pagerank_results = venturepulse_graph.pageRank(resetProbability=0.15, maxIter=10)

print("=" * 70)
print("TOP 15 MOST INFLUENTIAL INVESTORS (PageRank)")
print("=" * 70)
influential_investors = pagerank_results.vertices \
    .filter(F.col("type") == "investor") \
    .select("name", F.col("pagerank").alias("influence_score")) \
    .orderBy(F.desc("influence_score"))

influential_investors.limit(15).show(truncate=False)

print("\n💡 Insight: These investors should be prioritized for partnership outreach")

TOP 15 MOST INFLUENTIAL INVESTORS (PageRank)


+-----------------+------------------+
|name             |influence_score   |
+-----------------+------------------+
|BetaGrowth       |5.684499640184994 |
|NextLabs         |5.3059853787799955|
|StellarInnovation|4.331025315753262 |
|PeakHoldings     |4.121367763814845 |
|HorizonFund      |3.9902309375040885|
|BetaLabs         |3.794703980805315 |
|ApexEquity       |3.7775250301092185|
|GammaCapital     |3.7735158798988375|
|DeltaGroup       |3.7621385353728476|
|ZenithInnovation |3.7471280274360166|
|BetaGrowth       |3.71587846671573  |
|Future Investment|3.656327209194727 |
|NexusVentures    |3.5796468115037805|
|PrimeFund        |3.400169508709946 |
|PeakCollective   |3.3618073224998923|
+-----------------+------------------+


💡 Insight: These investors should be prioritized for partnership outreach


### 10.3 Label Propagation: Discover Investment Communities

**Question:** Are there natural communities or clusters in the investment network?

**Algorithm:** Label Propagation (community detection)

**Business Value:** Identify investment ecosystems (e.g., AI-focused community, HealthTech community)

In [20]:
# Run Label Propagation
communities = venturepulse_graph.labelPropagation(maxIter=5)

print("=" * 70)
print("INVESTMENT COMMUNITIES DETECTED")
print("=" * 70)

# Count communities
num_communities = communities.select("label").distinct().count()
print(f"\nNumber of communities: {num_communities}")

# Community sizes
community_sizes = communities.groupBy("label") \
    .agg(F.count("*").alias("size")) \
    .orderBy(F.desc("size"))

print("\nTop Communities by Size:")
community_sizes.show(10)

# Analyze largest community
largest_community_label = community_sizes.first()["label"]
largest_community = communities.filter(F.col("label") == largest_community_label)

print(f"\n" + "=" * 70)
print(f"LARGEST COMMUNITY ANALYSIS (Label: {largest_community_label})")
print("=" * 70)

# Sector distribution in largest community
print("\nSector Distribution:")
largest_community.filter(F.col("type") == "startup") \
    .groupBy("sector") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

print("\n💡 Insight: Communities reveal specialized investment ecosystems")



INVESTMENT COMMUNITIES DETECTED



Number of communities: 167

Top Communities by Size:


+-------------+----+
|        label|size|
+-------------+----+
| 936302870528| 584|
| 712964571138| 277|
| 257698037764|  20|
|1047972020227|  17|
| 773094113282|  10|
|1606317768706|   7|
| 549755813897|   6|
|1614907703299|   6|
| 644245094401|   2|
| 146028888066|   1|
+-------------+----+
only showing top 10 rows




LARGEST COMMUNITY ANALYSIS (Label: 936302870528)

Sector Distribution:


[Stage 821:=================================================>   (189 + 2) / 201]

+----------+-----+
|    sector|count|
+----------+-----+
|HealthTech|   67|
|      SaaS|   61|
|   Fintech|   61|
|    Gaming|   60|
|      Web3|   60|
| CleanTech|   60|
|        AI|   58|
|E-commerce|   57|
|       IoT|   56|
|  Mobility|   44|
+----------+-----+


💡 Insight: Communities reveal specialized investment ecosystems


---

<a id='step11'></a>
## Step 11: Business Insights from Graph

### Summary of Key Findings

In [21]:
# Step 11: Business Insights from Graph

print("\n" + "=" * 80)
print(" " * 20 + "VENTUREPULSE NETWORK INSIGHTS")
print("=" * 80)

# Network statistics
total_vertices = venturepulse_graph.vertices.count()
total_edges = venturepulse_graph.edges.count()
num_investors = venturepulse_graph.vertices.filter(F.col("type") == "investor").count()
num_startups = venturepulse_graph.vertices.filter(F.col("type") == "startup").count()

print(f"\n📊 Network Statistics:")
print(f"   Total Nodes: {total_vertices}")
print(f"   └─ Investors: {num_investors}")
print(f"   └─ Startups: {num_startups}")
print(f"   Total Edges: {total_edges} (bidirectional)")
print(f"   Unique Matches: {total_edges // 2}")
print(f"   Avg Matches per Startup: {(total_edges // 2) / num_startups:.1f}")
print(f"   Avg Opportunities per Investor: {(total_edges // 2) / num_investors:.1f}")

# Calculate isolated nodes
isolated_investors = out_degrees.filter(F.col("outDegree") == 0).count()
isolated_startups = in_degrees.filter(F.col("inDegree") == 0).count()
total_isolated = isolated_investors + isolated_startups

# Network density (using unique edges)
network_density = ((total_edges // 2) / (num_investors * num_startups)) * 100

# Top insights
print(f"\n🎯 Key Insights:")
print(f"   1. Network density: {network_density:.2f}% of all possible connections exist")
print(f"   2. Community structure: {num_communities} distinct investment ecosystems")

if total_isolated == 0:
    print(f"   3. ✅ Perfect matching: All entities have at least one match")
else:
    print(f"   3. ⚠️  Isolated entities: {total_isolated} nodes with zero matches")
    print(f"      - Investors: {isolated_investors}")
    print(f"      - Startups: {isolated_startups}")

# Sector analysis - USE STARTUPS DATAFRAME INSTEAD
print(f"\n📈 Top Investment Sectors:")
sector_distribution = startups_df \
    .groupBy("sector") \
    .count() \
    .orderBy(F.desc("count"))

top_3_sectors = sector_distribution.limit(3).collect()
for i, row in enumerate(top_3_sectors, 1):
    print(f"   {i}. {row['sector']}: {row['count']} startups")

# Actionable recommendations
print(f"\n💡 Strategic Recommendations:")
print(f"   → Partner with top 15 influential investors (identified via PageRank)")
print(f"   → Host sector-specific events leveraging {num_communities} community clusters")

if total_isolated > 0:
    print(f"   → Review matching criteria for {total_isolated} isolated entities")
else:
    print(f"   → Optimize match quality (all entities currently matched)")

# Platform health
match_coverage = (1 - (total_isolated / total_vertices)) * 100
print(f"\n✅ Platform Health Metrics:")
print(f"   Match Coverage: {match_coverage:.1f}% of entities connected")
print(f"   Avg Investor Portfolio: {(total_edges // 2) / num_investors:.1f} startups")
print(f"   Avg Startup Options: {(total_edges // 2) / num_startups:.1f} investors")

print("\n" + "=" * 80)
print("✅ Part 2: Graph Processing Complete")
print("=" * 80)




                    VENTUREPULSE NETWORK INSIGHTS



📊 Network Statistics:
   Total Nodes: 1087
   └─ Investors: 500
   └─ Startups: 587
   Total Edges: 55646 (bidirectional)
   Unique Matches: 27823
   Avg Matches per Startup: 47.4
   Avg Opportunities per Investor: 55.6

🎯 Key Insights:
   1. Network density: 9.48% of all possible connections exist
   2. Community structure: 167 distinct investment ecosystems
   3. ✅ Perfect matching: All entities have at least one match

📈 Top Investment Sectors:
   1. IoT: 179 startups
   2. SaaS: 174 startups
   3. Fintech: 169 startups

💡 Strategic Recommendations:
   → Partner with top 15 influential investors (identified via PageRank)
   → Host sector-specific events leveraging 167 community clusters
   → Optimize match quality (all entities currently matched)

✅ Platform Health Metrics:
   Match Coverage: 100.0% of entities connected
   Avg Investor Portfolio: 55.6 startups
   Avg Startup Options: 47.4 investors

✅ Part 2: Graph Processing Complete

Ready for Part 3: Machine Learning
  - Featur

<a id='part3'></a>
# Part 3: Machine Learning

## Objective

Build predictive models to identify high-potential startups and optimize investor matching.

**Business Goal:**  
Predict startup success probability to help investors prioritize their deal flow and startups identify their strongest matches.

**Approach:**  
Binary classification using startup features + graph metrics to predict success likelihood.

**Success Definition:**  
A startup is considered "successful" if it meets criteria indicating strong traction:
- Funding amount >= $10M (demonstrates market validation)
- Team size >= 50 (indicates scaling)
- Funding stage >= Series B (proven business model)

**This part demonstrates:** Feature engineering, ML pipelines, model training/evaluation, business insights from predictions.

---

<a id='step12'></a>
## Step 12: ML Setup & Feature Engineering

Prepare data for machine learning by creating features and defining target variable.

In [24]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Load startup data
ml_data = spark.read.parquet("/tmp/venturepulse/all_startup_events")
ml_data = ml_data.dropDuplicates(["startup_id"])

print(f"\n✅ Loaded {ml_data.count()} unique startups")

# Show schema
print("\nDataset Schema:")
ml_data.printSchema()



✅ Loaded 587 unique startups

Dataset Schema:
root
 |-- eventTime: timestamp (nullable = true)
 |-- event_category: string (nullable = true)
 |-- event_id: integer (nullable = true)
 |-- startup_id: string (nullable = true)
 |-- startup_name: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- funding_stage: string (nullable = true)
 |-- funding_amount_usd: long (nullable = true)
 |-- location: string (nullable = true)
 |-- team_size: integer (nullable = true)
 |-- months_since_founded: integer (nullable = true)
 |-- revenue_usd: long (nullable = true)
 |-- monthly_recurring_revenue: long (nullable = true)
 |-- team_growth_rate_6mo: double (nullable = true)
 |-- event_type: string (nullable = true)



In [25]:
# Define success based on traction metrics (NOT on features we'll use!)
ml_data = ml_data.withColumn(
    "high_traction",
    F.when(
        (F.col("team_growth_rate_6mo") >= 0.25) &  # 25%+ growth
        (F.col("revenue_usd") >= 500000) &  # $500K+ revenue
        (F.col("monthly_recurring_revenue") >= 25000),  # $25K+ MRR
        1.0
    ).otherwise(0.0)
)

# Show distribution
print("\n📊 Target Variable Distribution:")
ml_data.groupBy("high_traction").count().show()

success_rate = ml_data.filter(F.col("high_traction") == 1.0).count() / ml_data.count() * 100
print(f"High-traction rate: {success_rate:.1f}%\n")


📊 Target Variable Distribution:
+-------------+-----+
|high_traction|count|
+-------------+-----+
|          0.0|  401|
|          1.0|  186|
+-------------+-----+

High-traction rate: 31.7%



In [26]:
# Create features for ML
ml_features = ml_data.select(
    "startup_id",
    "startup_name",
    "sector",
    "funding_stage",
    "funding_amount_usd",
    "team_size",
    "months_since_founded",
    "revenue_usd",
    "monthly_recurring_revenue",
    "team_growth_rate_6mo",
    "high_traction"
).na.drop()  # Remove rows with null values

print(f"\n✅ Feature dataset: {ml_features.count()} startups after removing nulls")

# Encode categorical variables
sector_indexer = StringIndexer(inputCol="sector", outputCol="sector_index")
stage_indexer = StringIndexer(inputCol="funding_stage", outputCol="stage_index")

# Select numeric features
feature_cols = [
    "sector_index",
    "stage_index", 
    "funding_amount_usd",
    "team_size",
    "months_since_founded",
]

# Assemble features into vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")

# Scale features
scaler = StandardScaler(inputCol="features_raw", outputCol="features")

print("\n✅ Feature engineering pipeline configured")
print(f"   Features: {len(feature_cols)}")


✅ Feature dataset: 587 startups after removing nulls

✅ Feature engineering pipeline configured
   Features: 5


---

<a id='step13'></a>
## Step 13: Success Prediction Model

Train and evaluate machine learning models to predict startup success.

In [27]:
print("=" * 70)
print("STEP 13: MODEL TRAINING & EVALUATION")
print("=" * 70)

# Split data
train_data, test_data = ml_features.randomSplit([0.8, 0.2], seed=42)

print(f"\n📊 Dataset Split:")
print(f"   Training: {train_data.count()} startups")
print(f"   Testing: {test_data.count()} startups")

train_success_rate = train_data.filter(F.col("high_traction") == 1).count() / train_data.count() * 100
test_success_rate = test_data.filter(F.col("high_traction") == 1).count() / test_data.count() * 100

print(f"\n   Training success rate: {train_success_rate:.1f}%")
print(f"   Testing success rate: {test_success_rate:.1f}%")

STEP 13: MODEL TRAINING & EVALUATION

📊 Dataset Split:
   Training: 496 startups


   Testing: 91 startups



   Training success rate: 30.8%
   Testing success rate: 36.3%


In [28]:
print("\n" + "=" * 70)
print("MODEL 1: LOGISTIC REGRESSION")
print("=" * 70)

# Logistic Regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="high_traction",
    maxIter=100
)

# Create pipeline
lr_pipeline = Pipeline(stages=[
    sector_indexer,
    stage_indexer,
    assembler,
    scaler,
    lr
])

# Train model
print("\n⏳ Training Logistic Regression...")
lr_model = lr_pipeline.fit(train_data)
print("✅ Model trained")

# Make predictions
lr_predictions = lr_model.transform(test_data)

# Evaluate
evaluator_auc = BinaryClassificationEvaluator(labelCol="high_traction", metricName="areaUnderROC")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="high_traction", metricName="accuracy")

lr_auc = evaluator_auc.evaluate(lr_predictions)
lr_accuracy = evaluator_acc.evaluate(lr_predictions)

print(f"\n📊 Logistic Regression Performance:")
print(f"   AUC-ROC: {lr_auc:.3f}")
print(f"   Accuracy: {lr_accuracy:.3f}")

# Confusion Matrix
print("\nConfusion Matrix:")
lr_predictions.groupBy("high_traction", "prediction").count().show()


MODEL 1: LOGISTIC REGRESSION

⏳ Training Logistic Regression...


26/03/07 19:28:42 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/07 19:28:42 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


✅ Model trained

📊 Logistic Regression Performance:
   AUC-ROC: 0.700
   Accuracy: 0.681

Confusion Matrix:


[Stage 1015:>                                                       (0 + 1) / 1]

+-------------+----------+-----+
|high_traction|prediction|count|
+-------------+----------+-----+
|          1.0|       1.0|   12|
|          0.0|       1.0|    8|
|          1.0|       0.0|   21|
|          0.0|       0.0|   50|
+-------------+----------+-----+



In [29]:
print("\n" + "=" * 70)
print("MODEL 2: RANDOM FOREST")
print("=" * 70)

# Random Forest model
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="high_traction",
    numTrees=100,
    seed=42
)

# Create pipeline
rf_pipeline = Pipeline(stages=[
    sector_indexer,
    stage_indexer,
    assembler,
    scaler,
    rf
])

# Train model
print("\n⏳ Training Random Forest...")
rf_model = rf_pipeline.fit(train_data)
print("✅ Model trained")

# Make predictions
rf_predictions = rf_model.transform(test_data)

# Evaluate
rf_auc = evaluator_auc.evaluate(rf_predictions)
rf_accuracy = evaluator_acc.evaluate(rf_predictions)

print(f"\n📊 Random Forest Performance:")
print(f"   AUC-ROC: {rf_auc:.3f}")
print(f"   Accuracy: {rf_accuracy:.3f}")

# Confusion Matrix
print("\nConfusion Matrix:")
rf_predictions.groupBy("high_traction", "prediction").count().show()

# Feature Importance
print("\n📊 Top 5 Important Features:")
feature_importance = rf_model.stages[-1].featureImportances
importance_list = [(feature_cols[i], float(feature_importance[i])) 
                   for i in range(len(feature_cols))]
importance_list.sort(key=lambda x: x[1], reverse=True)

for feature, importance in importance_list[:5]:
    print(f"   {feature}: {importance:.3f}")


MODEL 2: RANDOM FOREST

⏳ Training Random Forest...


✅ Model trained



📊 Random Forest Performance:
   AUC-ROC: 0.746
   Accuracy: 0.670

Confusion Matrix:
+-------------+----------+-----+
|high_traction|prediction|count|
+-------------+----------+-----+
|          1.0|       1.0|    7|
|          0.0|       1.0|    4|
|          1.0|       0.0|   26|
|          0.0|       0.0|   54|
+-------------+----------+-----+


📊 Top 5 Important Features:
   team_size: 0.342
   funding_amount_usd: 0.336
   months_since_founded: 0.158
   sector_index: 0.114
   stage_index: 0.050


In [30]:
print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(f"\n{'Model':<20} {'AUC-ROC':<10} {'Accuracy':<10}")
print("-" * 40)
print(f"{'Logistic Regression':<20} {lr_auc:<10.3f} {lr_accuracy:<10.3f}")
print(f"{'Random Forest':<20} {rf_auc:<10.3f} {rf_accuracy:<10.3f}")

best_model_name = "Random Forest" if rf_auc > lr_auc else "Logistic Regression"
best_auc = max(rf_auc, lr_auc)

print(f"\n🏆 Best Model: {best_model_name} (AUC: {best_auc:.3f})")


MODEL COMPARISON

Model                AUC-ROC    Accuracy  
----------------------------------------
Logistic Regression  0.700      0.681     
Random Forest        0.746      0.670     

🏆 Best Model: Random Forest (AUC: 0.746)


---

<a id='step14'></a>
## Step 14: Business Insights from ML

Generate actionable insights from model predictions.

In [31]:
print("=" * 70)
print("BUSINESS INSIGHTS FROM MACHINE LEARNING")
print("=" * 70)

# Use best model predictions (Random Forest)
predictions = rf_predictions

# Convert probability vector to array, then extract the second element
# This uses Spark's built-in vector_to_array function for ML vectors
from pyspark.ml.functions import vector_to_array

predictions_with_prob = predictions.withColumn(
    "probability_array", 
    vector_to_array(F.col("probability"))
).withColumn(
    "success_probability",
    F.col("probability_array")[1]  # Get index 1 (second element - success probability)
).drop("probability_array")  # Clean up temporary column

# High-potential startups (predicted successful)
high_potential = predictions_with_prob.filter(F.col("prediction") == 1.0)

print(f"\n🎯 High-Potential Startups Identified: {high_potential.count()}")

print("\nTop Extreme High-Potential Startups:")
high_potential.select(
    "startup_name",
    "sector",
    "funding_stage",
    "team_size",
    F.round("success_probability", 3).alias("success_probability")
).orderBy(F.desc("success_probability")).show(10, truncate=False)

# Analyze by sector
print("\n📊 High-Potential Startups by Sector:")
high_potential.groupBy("sector") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

# Analyze by funding stage
print("\n📊 High-Potential Startups by Stage:")
high_potential.groupBy("funding_stage") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

# False Negatives (actual success but predicted failure)
false_negatives = predictions_with_prob.filter(
    (F.col("high_traction") == 1.0) & (F.col("prediction") == 0.0)
)

print(f"\n⚠️  Potentially Undervalued Startups: {false_negatives.count()}")
print("   (Successful startups the model didn't identify - hidden gems)")

if false_negatives.count() > 0:
    print("\nTop 5 Undervalued Startups:")
    false_negatives.select(
        "startup_name",
        "sector",
        "funding_amount_usd",
        "team_size",
        F.round("success_probability", 3).alias("success_probability")
    ).orderBy(F.desc("success_probability")).show(5, truncate=False)

# Risky investments (predicted to fail but actually succeeded - close calls)
print("\n💎 Hidden Gems Analysis:")
risky_but_successful = predictions_with_prob.filter(
    (F.col("high_traction") == 1.0) & 
    (F.col("prediction") == 0.0) &
    (F.col("success_probability") > 0.3)  # Close calls with >30% probability
)
print(f"   Startups that succeeded despite low predicted probability: {risky_but_successful.count()}")

# Average probability by sector
print("\n📈 Average Success Probability by Sector:")
predictions_with_prob.groupBy("sector") \
    .agg(
        F.round(F.avg("success_probability"), 3).alias("avg_success_prob"),
        F.count("*").alias("total_startups")
    ) \
    .orderBy(F.desc("avg_success_prob")) \
    .show(truncate=False)

# Average probability by funding stage
print("\n📈 Average Success Probability by Funding Stage:")
predictions_with_prob.groupBy("funding_stage") \
    .agg(
        F.round(F.avg("success_probability"), 3).alias("avg_success_prob"),
        F.count("*").alias("total_startups")
    ) \
    .orderBy(F.desc("avg_success_prob")) \
    .show(truncate=False)

# Distribution of predictions
print("\n📊 Prediction Distribution:")
predictions_with_prob.groupBy("prediction") \
    .count() \
    .withColumn("percentage", F.round((F.col("count") / predictions_with_prob.count() * 100), 1)) \
    .show()

# Probability distribution buckets
print("\n📊 Success Probability Distribution:")
predictions_with_prob.withColumn(
    "prob_bucket",
    F.when(F.col("success_probability") < 0.3, "Low (<30%)")
     .when(F.col("success_probability") < 0.5, "Medium (30-50%)")
     .when(F.col("success_probability") < 0.7, "High (50-70%)")
     .otherwise("Very High (>70%)")
).groupBy("prob_bucket") \
    .count() \
    .orderBy("prob_bucket") \
    .show()

# Key insights
print("\n" + "=" * 70)
print("💡 KEY INSIGHTS & RECOMMENDATIONS")
print("=" * 70)

print(f"\n1. Model Performance:")
print(f"   - Best Model: Random Forest")
print(f"   - AUC-ROC: {rf_auc:.3f} (Good discrimination capability)")
print(f"   - Accuracy: {rf_accuracy:.3f}")

print(f"\n2. Feature Importance:")
print(f"   - Top driver: {importance_list[0][0]} ({importance_list[0][1]:.3f})")
print(f"   - Second: {importance_list[1][0]} ({importance_list[1][1]:.3f})")
print(f"   - Third: {importance_list[2][0]} ({importance_list[2][1]:.3f})")
print(f"   → Financial metrics and team characteristics are strong success predictors")

print(f"\n3. Investment Opportunities:")
high_potential_count = high_potential.count()
false_negative_count = false_negatives.count()
total_predictions = predictions_with_prob.count()
print(f"   - {high_potential_count} high-potential startups identified ({high_potential_count/total_predictions*100:.1f}% of dataset)")
print(f"   - {false_negative_count} potentially undervalued opportunities (hidden gems)")

# Get top sector
top_sector_row = high_potential.groupBy("sector").count().orderBy(F.desc("count")).first()
if top_sector_row:
    print(f"   - Focus sector: {top_sector_row['sector']} ({top_sector_row['count']} high-potential startups)")

# Get top funding stage
top_stage_row = high_potential.groupBy("funding_stage").count().orderBy(F.desc("count")).first()
if top_stage_row:
    print(f"   - Focus stage: {top_stage_row['funding_stage']} ({top_stage_row['count']} high-potential startups)")

print(f"\n4. Platform Strategy:")
print(f"   - Use predictions to prioritize investor notifications")
print(f"   - Highlight high-probability startups (>70% success prob) in investor dashboards")
print(f"   - Create 'Hidden Gems' section for undervalued startups with probability >30%")
print(f"   - Segment investors by sector preference using these insights")

print(f"\n5. Risk Mitigation:")
print(f"   - Model accuracy of {rf_accuracy:.1%} means ~{(1-rf_accuracy)*100:.0f}% of predictions may be incorrect")
print(f"   - Combine ML predictions with human due diligence")
print(f"   - Focus on probability scores rather than binary predictions")
print(f"   - Regular model retraining recommended as new data arrives")

print("\n" + "=" * 70)
print("✅ Part 3: Machine Learning - Business Insights Complete")
print("=" * 70)

BUSINESS INSIGHTS FROM MACHINE LEARNING



🎯 High-Potential Startups Identified: 11

Top Extreme High-Potential Startups:
+--------------+----------+-------------+---------+-------------------+
|startup_name  |sector    |funding_stage|team_size|success_probability|
+--------------+----------+-------------+---------+-------------------+
|QuantumTech   |HealthTech|Series B     |133      |0.701              |
|MetaVertex    |CleanTech |Series C     |297      |0.586              |
|CyberPlatform |HealthTech|Series C     |160      |0.575              |
|EcoStream     |IoT       |Series B     |147      |0.575              |
|DataPlatform  |HealthTech|Series C     |102      |0.567              |
|MetaAnalytics |IoT       |Series C     |148      |0.556              |
|HyperNetwork  |SaaS      |Series C     |233      |0.554              |
|BetaFlow      |HealthTech|Series C     |101      |0.53               |
|EcoInnovations|E-commerce|Series C     |188      |0.522              |
|DigiSync      |IoT       |Series C     |80       |0.522

+-------------+-----+
|funding_stage|count|
+-------------+-----+
|Series C     |9    |
|Series B     |2    |
+-------------+-----+


⚠️  Potentially Undervalued Startups: 26
   (Successful startups the model didn't identify - hidden gems)

Top 5 Undervalued Startups:
+-------------+----------+------------------+---------+-------------------+
|startup_name |sector    |funding_amount_usd|team_size|success_probability|
+-------------+----------+------------------+---------+-------------------+
|RapidWorks   |E-commerce|113365526         |163      |0.492              |
|TechAnalytics|IoT       |38092218          |90       |0.487              |
|RapidWorks   |Mobility  |45079712          |77       |0.482              |
|CoreSystems  |E-commerce|43670236          |170      |0.478              |
|DataLabs     |E-commerce|141916237         |213      |0.477              |
+-------------+----------+------------------+---------+-------------------+
only showing top 5 rows


💎 Hidden Gems Analysi

   - Focus sector: HealthTech (4 high-potential startups)
   - Focus stage: Series C (9 high-potential startups)

4. Platform Strategy:
   - Use predictions to prioritize investor notifications
   - Highlight high-probability startups (>70% success prob) in investor dashboards
   - Create 'Hidden Gems' section for undervalued startups with probability >30%
   - Segment investors by sector preference using these insights

5. Risk Mitigation:
   - Model accuracy of 67.0% means ~33% of predictions may be incorrect
   - Combine ML predictions with human due diligence
   - Focus on probability scores rather than binary predictions
   - Regular model retraining recommended as new data arrives

✅ Part 3: Machine Learning - Business Insights Complete
